# 06 - Régression linéaire

On estime l’effet du contexte social et du retard scolaire sur le score de lecture.
La formule de base est :
score_lecture ~ pcs + sexe + retard


In [1]:
from pathlib import Path
import sys
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.loader import load_csv, data_path

df = load_csv(data_path('data', 'interim', 'etude_lecture_6e_clean.csv'))

df['sexe'] = df['sexe'].map({'F': 0, 'M': 1})
df['retard'] = df['retard'].astype(int)

dummy_pcs = pd.get_dummies(df['pcs'], prefix='pcs', drop_first=True).astype(float)
model_df = pd.concat([df[['score_lecture', 'sexe', 'retard']], dummy_pcs], axis=1)
X = model_df.drop(columns=['score_lecture'])
y = model_df['score_lecture']
X = sm.add_constant(X)
# Ensure exogenous variables are numeric (coerce any unexpected object dtype)
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
# Ensure target is numeric and align indices
y = pd.to_numeric(y, errors='coerce')
# Drop rows with NaNs just in case
valid_idx = X.index[~(X.isnull().any(axis=1) | y.isnull())]
X = X.loc[valid_idx]
y = y.loc[valid_idx]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:          score_lecture   R-squared:                       0.541
Model:                            OLS   Adj. R-squared:                  0.540
Method:                 Least Squares   F-statistic:                     2521.
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:27:04   Log-Likelihood:                -55850.
No. Observations:               15000   AIC:                         1.117e+05
Df Residuals:                   14992   BIC:                         1.118e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       